In [ ]:
import torch
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
from pathlib import Path
import csv, json

# gpu + experimental settings
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, TAU, ALPHA = 1000, 0.5, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)
dataset = "hiv"

METHOD = "ppat"
LAMBDA_INIT, LAMBDA_UPDATE_EVERY = 0.5, 100

# directories
DATA_DIR = Path(f"../datasets/{dataset}")
LOG_DIR = Path("../logs")
fname = f"{dataset}_{METHOD}.csv"
log_path = LOG_DIR / fname

# keep the original per-seed CSV schema, so existing analysis code can read it.
columns = ["dataset", "budget", "tau", "method", "seed", "thetahat", "coverage", "width", "n_labels"]
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Use a fresh LOG_DIR, or explicitly set True to replace this method's output.
OVERWRITE = False

In [ ]:
#### LOAD IN OUR DATA

# load in the standardized data
Xs = torch.load(DATA_DIR / "X_test.pt", map_location="cpu", weights_only=True).to(device)
ys = torch.load(DATA_DIR / "y_test.pt", map_location="cpu", weights_only=True).to(device)
beta_init = torch.load(DATA_DIR / "beta_init.pt", map_location="cpu", weights_only=True).to(device)
N, D = Xs.shape

if ys.shape != (N,) or beta_init.shape != (D,):
    raise ValueError("Expected X_test: (N,D), y_test: (N,), beta_init: (D,).")
if any(x.dtype != torch.float32 for x in (Xs, ys, beta_init)):
    raise ValueError("Keep all three inputs float32, as in the supplied PAI experiment.")
if not all(torch.isfinite(x).all().item() for x in (Xs, ys, beta_init)):
    raise ValueError("Input tensors must be finite.")
if not ((ys == 0) | (ys == 1)).all().item():
    raise ValueError("This notebook specializes PPAT to a binary-outcome mean.")
if N_SEEDS < 2 or not (0 < TAU <= 1) or not (0 < ALPHA < 1):
    raise ValueError("Require N_SEEDS >= 2, 0 < TAU <= 1, and 0 < ALPHA < 1.")
if not all(2 <= int(b * N) < N for b in BUDGET_PROPS):
    raise ValueError("Every budget must give 2 <= n < N for the reported CIs.")

# PPAT Section 3, p.4: FIXED proxy array and its FULL-POOL mean.
# The online model below is the acquisition surrogate, not a replacement for cs.
# beta_init is assumed to come from permissible initial/training data.
with torch.no_grad():
    proxy_fxs = torch.sigmoid(beta_init @ Xs.T).to(torch.float64)  # (N,)
    proxy_mean = proxy_fxs.mean()
    cs = proxy_fxs - proxy_mean                                 # (N,), fixed
c_for_sampling = cs.to(torch.float32).unsqueeze(0)                # (1,N)
proxy_second_moment = cs.square().mean()                     # N^{-1} sum_i c_i^2
proxy_degenerate = (proxy_second_moment.item() == 0.0)
if proxy_degenerate:
    print("Constant proxy: lambda_hat is undefined; using lambda=0 (exactly LURE).")
if LAMBDA_UPDATE_EVERY < 1:
    raise ValueError("LAMBDA_UPDATE_EVERY must be positive.")

# Ground truth is used ONLY for evaluation, never for proposals or lambda fitting.
# Preserve the original float32 definition for the logged coverage metric.
true_theta = ys.mean(dim=-1, keepdim=True)
print(f"{METHOD}: N={N:,}, D={D}, seeds={N_SEEDS}, device={device}")
print("Budgets:", [(float(b), int(b * N)) for b in BUDGET_PROPS])
print(f"Nominal coverage={1-ALPHA:.0%}; uniform mixing={TAU}; proxy is fixed.")

In [ ]:
#### PREDICTION-POWERED ACTIVE TESTING

# Start a clean run; do not append duplicate seed/budget rows to a previous run.
if log_path.exists() and log_path.stat().st_size > 0 and not OVERWRITE:
    raise FileExistsError(f"{log_path} already exists. Use a new LOG_DIR or set OVERWRITE=True.")
with open(log_path, "w", newline="") as file:
    csv.writer(file).writerow(columns)

# Small provenance sidecar; the per-seed CSV stays unchanged.
run_config = {
    "method": METHOD, "dataset": dataset, "source": "PPAT arXiv:2607.08347v1",
    "N": N, "D": D, "n_seeds": N_SEEDS, "tau": TAU, "alpha": ALPHA,
    "budget_props": BUDGET_PROPS.tolist(),
    "label_budgets": [int(b * N) for b in BUDGET_PROPS],
    "seed_rule": "858 + budget_idx; one separately restarted batched run per budget",
    "estimand": "mean(y_test), not classifier accuracy or cross-entropy",
    "proxy": "fixed sigmoid(X_test @ beta_init), centered over the full pool",
    "surrogate": "original diagonal-precision online logistic update after every label",
    "acquisition": "PPAT Eq.12 Bernoulli RMS, Eq.13 normalization + uniform mixing",
    "inference": "Eq.8 and Theorem E.15 / Corollary E.19; no PAI variance correction",
    "model_dtype": "float32", "estimator_dtype": "float64",
    "device": str(device), "torch_version": torch.__version__,
    "lambda": {"type": "Eq.14 plugin", "initial": LAMBDA_INIT, "acquisition_update_every": LAMBDA_UPDATE_EVERY, "final_refit": True, "clipping": False},
}
with open(log_path.with_suffix(".json"), "w") as file:
    json.dump(run_config, file, indent=2)

# Only for a compact summary of the coefficients actually used at each endpoint.
lambda_summaries = []

with torch.inference_mode():
    # Same budget loop, independent budget restarts, and RNG seed rule as PAI.
    for budget_idx, budget in enumerate(BUDGET_PROPS):
        n = int(budget * N)       # Paper's final budget M; NOT rounded up.
        torch.random.manual_seed(858 + budget_idx)

        #### initialize data structures
        already_sampled = torch.zeros((N_SEEDS, N), dtype=bool, device=device)
        betas = torch.tile(beta_init, (N_SEEDS, 1))
        diag_precisions = torch.full((N_SEEDS, D), 1.0, device=device, dtype=torch.float32)

        # Keep only queried indices and their actual, mixed sampling probabilities.
        # Final lambda must be applied to ALL terms; do not store time-varying residuals.
        I_history = torch.empty((N_SEEDS, n), dtype=torch.long, device=device)
        q_history = torch.empty((N_SEEDS, n), dtype=torch.float64, device=device)
        # Section 5.1 (p.9): initialize at 0.5; subsequent acquisition updates every 100 labels.
        # An exactly constant proxy has c=0, so PPAT is LURE; lambda=0 avoids 0/0.
        lambda_acq = torch.full((N_SEEDS, 1), 0.0 if proxy_degenerate else LAMBDA_INIT,
                                device=device, dtype=torch.float64)

        #### adaptive sampling rounds
        for t in tqdm(range(n), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}"):

            #### PART 1: PPAT acquisition, Eq.12-13 (p.6)
            logits = betas @ Xs.T                     # (N_SEEDS, N)
            fxs = torch.sigmoid(logits)                # current surrogate probabilities

            # For Y~Bernoulli(p), a^2 = p(1-p) + (p - lambda*c)^2.
            # hypot(a,b) computes sqrt(a^2+b^2), avoiding overflow from squaring lambda.
            # Surrogate/sampling arithmetic stays float32; inference below uses float64.
            shift = lambda_acq.to(torch.float32) * c_for_sampling
            scores = torch.hypot(torch.sqrt(fxs * (1.0 - fxs)), fxs - shift)
            log_u_ts = scores.log().masked_fill(already_sampled, -torch.inf)

            # Undefined zero-score normalization: use uniform on remaining points.
            # This is only a degenerate numerical safeguard, not a score floor/tuning knob.
            zero_score_rows = torch.isneginf(log_u_ts).all(dim=1, keepdim=True)
            log_u_ts = torch.where(zero_score_rows & ~already_sampled, 0.0, log_u_ts)

            h_js = torch.softmax(log_u_ts, dim=1)
            q_js = (TAU * ~already_sampled / (N - t)) + ((1.0 - TAU) * h_js)
            q_js = q_js / q_js.sum(dim=1, keepdim=True)  # explicit numerical normalization
            I_t = torch.multinomial(input=q_js, num_samples=1)

            #### PART 2: record queried labels / probabilities for PPAT
            y_Is = ys[I_t]                                        # (N_SEEDS, 1)
            f_Is = torch.gather(fxs, dim=1, index=I_t)
            q_Is = torch.gather(q_js, dim=1, index=I_t)
            I_history[:, t:t+1] = I_t
            q_history[:, t:t+1] = q_Is.to(torch.float64)
            already_sampled.scatter_(dim=1, index=I_t,
                src=torch.full_like(I_t, fill_value=True, dtype=bool))

            #### PART 3: ORIGINAL online logistic-regression update (unchanged)
            w_Is = f_Is * (1.0 - f_Is)
            X_Is = Xs[I_t.flatten()]
            diag_precisions += w_Is * (X_Is ** 2)
            betas += (y_Is - f_Is) * X_Is / diag_precisions

            #### PART 3b: update lambda for FUTURE sampling only
            m = t + 1
            if m % LAMBDA_UPDATE_EVERY == 0 and m < n and not proxy_degenerate:
                # Eq.14 (p.7) at CURRENT budget M=m, not the final budget n.
                # The earlier q_s(I_s) are the probabilities actually used at those draws.
                past_ms = torch.arange(1, m + 1, device=device, dtype=torch.float64).unsqueeze(0)
                V_current = 1.0 + ((N - m) / (N - past_ms)) * (
                    1.0 / ((N - past_ms + 1.0) * q_history[:, :m]) - 1.0)
                past_indices = I_history[:, :m]
                numerator = (V_current * ys[past_indices].to(torch.float64)
                             * cs[past_indices]).mean(dim=1, keepdim=True)
                lambda_acq = numerator / proxy_second_moment
                if not torch.isfinite(lambda_acq).all().item():
                    raise FloatingPointError("Nonfinite lambda estimate; no coefficient clipping is used.")


        #### PART 4: PPAT point estimate and PPAT confidence interval
        # m is ONE-indexed in the paper, whereas the sampling loop's t is zero-indexed.
        ms = torch.arange(1, n + 1, device=device, dtype=torch.float64).unsqueeze(0)
        y_history = ys[I_history].to(torch.float64)     # ONLY queried labels
        c_history = cs[I_history]                       # FIXED centered proxy

        # LURE weights: Eq.5 (p.3), equivalently Eq.22 (p.38), using final M=n.
        Vs = 1.0 + ((N - n) / (N - ms)) * (
            1.0 / ((N - ms + 1.0) * q_history) - 1.0)
        # Eq.14 and Section E.5.4: refit lambda_M on ALL n queried labels at this endpoint.
        # Do this even when n is not divisible by 100. Cadence governs acquisition only.
        if proxy_degenerate:
            lambda_final = torch.zeros((N_SEEDS, 1), device=device, dtype=torch.float64)
        else:
            lambda_final = (Vs * y_history * c_history).mean(dim=1, keepdim=True) / proxy_second_moment
        # No [0,1] clipping, ridge, or per-round-lambda substitution is in the paper's Eq.14.

        # Eq.8 (p.4): original PPAT, NOT augmented LURE or equal-weight PAI.
        residuals = y_history - lambda_final * c_history
        thetahats = (Vs * residuals).mean(dim=1, keepdim=True)

        # Lemma E.4 (p.38): A_m includes previously queried residuals.
        # Use the SAME FINAL lambda in every current and previous residual.
        prefix_residuals = torch.cumsum(residuals, dim=1) - residuals
        A_ms = (residuals / q_history + prefix_residuals) / N
        gammas = N * (N - n) / ((N - ms) * (N - ms + 1.0))

        # Theorem E.15 (p.52); plug-in case: Corollary E.19 (p.59).
        # This estimates the CLT variance; the estimated Var(thetahat) is sigmahats_sq/n.
        sigmahats_sq = ((gammas * (A_ms - thetahats)) ** 2).mean(dim=1, keepdim=True)

        # Cheap endpoint checks: the two EXACT representations must agree.
        torch.testing.assert_close(thetahats, (gammas * A_ms).mean(dim=1, keepdim=True),
                                   rtol=1e-9, atol=1e-11)
        assert (already_sampled.sum(dim=1) == n).all().item(), "Repeated label detected."
        if not (torch.isfinite(thetahats).all() & torch.isfinite(sigmahats_sq).all()).item():
            raise FloatingPointError("Nonfinite PPAT estimate/variance; no result was silently clipped.")

        # Same ALPHA and untruncated Wald intervals
        z_score = norm.ppf(1 - (ALPHA / 2))
        ubs = thetahats + z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
        lbs = thetahats - z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
        widths = (ubs - lbs).cpu().numpy()
        coverages = ((lbs <= true_theta) & (true_theta <= ubs)).cpu().numpy()
        thetahats = thetahats.cpu().numpy()

        # log results
        results = [
            [dataset, budget, TAU, METHOD, seed,
             thetahats[seed][0], coverages[seed][0], widths[seed][0], n]
            for seed in range(N_SEEDS)]
        with open(log_path, "a", newline="") as file:
            csv.writer(file).writerows(results)

        final_lambdas = torch.as_tensor(lambda_final).detach().cpu().numpy().reshape(-1)
        lambda_summaries.append({"budget": float(budget),
            "lambda_mean": float(final_lambdas.mean()),
            "lambda_min": float(final_lambdas.min()),
            "lambda_max": float(final_lambdas.max())})
        print(f"budget={budget:.3f}, n={n}, coverage={coverages.mean():.3f}, "
              f"mean width={widths.mean():.6f}, lambda mean={final_lambdas.mean():.4f}")